# Post-hoc

Scores trained checkpoints. No training. Everything runs through
`scripts/posthoc.py`; each model is rebuilt from the config inside its own
checkpoint, so its resolution and encoding are reproduced exactly.

Order: **inspect → validation → TTA/thresholds → test, once.**

## 0. Colab web UI only — clone

Skip if `/content/fdl-project` exists.

In [ ]:
from getpass import getpass
from pathlib import Path
import subprocess

TARGET = Path("/content/fdl-project")
BRANCH = "feature/phase3-architectures"
REMOTE = "github.com/ezero3/fdl-project.git"


def run(*command: str) -> None:
    subprocess.run(command, check=True)


if TARGET.exists():
    print(f"{TARGET} already present -- pulling")
    run("git", "-C", str(TARGET), "fetch", "origin", BRANCH)
    run("git", "-C", str(TARGET), "checkout", BRANCH)
    run("git", "-C", str(TARGET), "pull", "--ff-only")
else:
    # Private repo, so the clone needs a personal access token. getpass keeps it
    # out of the notebook and out of the output.
    token = getpass("GitHub personal access token (input hidden): ").strip()
    run("git", "clone", "--branch", BRANCH,
        f"https://{token}@{REMOTE}", str(TARGET))
    # Drop the token from the stored remote; a later pull will ask again rather
    # than leaving a credential sitting in .git/config.
    run("git", "-C", str(TARGET), "remote", "set-url", "origin", f"https://{REMOTE}")
    del token

print(subprocess.run(["git", "-C", str(TARGET), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip())

## 1. Setup

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")   # idempotent; never force_remount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl", DATASET)
assert DATASET.stat().st_size == EXPECTED_BYTES, "wrong pickle: splits are row indices"

CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)

print(f"gpu     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {DATASET.stat().st_size / 1024**3:.2f} GiB")

## 2. Inspect

What each checkpoint actually is — run name, model, resolution, epoch, best metric. No GPU.

Use it to resolve duplicate Drive folders: a re-run lands beside the original as
`<name> (1)`, and `best_epoch` / `best_metric` say which is which.

In [ ]:
import shutil, subprocess, sys

import torch
from pathlib import Path

CHECKPOINTS_DIR = CHECKPOINTS if HAS_DRIVE else REPO / "trained-models/checkpoints"
OUTPUT_ROOT = REPO / "output/posthoc"


def stream(command: list[str]) -> int:
    """Run a script and echo its output here, line by line as it arrives.

    Not inherited stdout: the VS Code Colab bridge does not forward a
    subprocess's stream to the cell, so the script runs correctly and shows
    nothing but its exit code. Not capture_output either: that would hold
    everything back until the end, and these runs take many minutes.
    """

    print(" ".join(command), "\n", flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    process.wait()
    if process.returncode:
        print(f"\n[exit {process.returncode}]")
    return process.returncode


def posthoc(*args: str) -> int:
    return stream([sys.executable, str(REPO / "scripts/posthoc.py"),
                   "--checkpoints-dir", str(CHECKPOINTS_DIR),
                   "--output-root", str(OUTPUT_ROOT), *args])


def find_checkpoint(run_name: str) -> Path:
    """The checkpoint for a run, tolerating Drive's duplicate folders.

    A re-run lands beside the original as `<name> (1)`. When both exist the one
    that trained further wins -- that is the completed run, and the other is the
    interrupted attempt.
    """

    candidates = sorted(CHECKPOINTS_DIR.glob(f"{run_name}*"))
    files = []
    for directory in candidates:
        for name in ("best.pt", "best_model.pt", "last.pt"):
            path = directory / name
            if path.is_file():
                files.append(path)
                break
    assert files, f"no checkpoint for {run_name!r} under {CHECKPOINTS_DIR}"
    if len(files) == 1:
        return files[0]
    best, best_epoch = None, -1
    for path in files:
        payload = torch.load(path, map_location="cpu", weights_only=False)
        epoch = int(payload.get("best_epoch") or payload.get("epoch") or 0)
        print(f"  {path.parent.name}: best_epoch={epoch} metric={payload.get('best_metric')}")
        if epoch > best_epoch:
            best, best_epoch = path, epoch
    print(f"  -> using {best.parent.name}")
    return best


posthoc("--inspect");

## 3. Validation

Edit `ONLY` to choose runs. Results append to `comparison.csv` after each model.

In [ ]:
# The runs that go in the presentation, plus the two the dilation claim needs.
ONLY = [
    "v32-resnet34_finetune",     # 0.9041  best overall, pretrained CNN
    "v32-vit_b_32_finetune",     # 0.9029  pretrained transformer
    "v28-convnext_big_128",      # 0.8988  best from scratch, ours
    "v31-dilated_rotation_64",   # 0.8852  dilated
    "v31-dilated_control_64",    # 0.8216  the same model with tight 3x3s
    "v31-convnext_dilated",      # 0.8957  dilation on ConvNeXt: no gain
]

flags = [flag for name in ONLY for flag in ("--only", name)]
posthoc("--split", "validation", *flags);

## 4. TTA and per-class thresholds

Both are free gains if they hold — no retraining. Compare against section 3.

In [ ]:
# TTA averages over the 8 square symmetries -- 8x the inference, no retraining.
# Thresholds are fitted on validation here and reused on test in the next cell.
posthoc("--split", "validation", "--tta", "--tune-thresholds",
        "--results", str(OUTPUT_ROOT / "comparison_tta.csv"), *flags);

## 5. Test — once

The protocol allows **one** test evaluation. Narrow `FINAL` to the models you are reporting
before running this; thresholds fitted in section 4 are reused rather than refitted.

In [ ]:
# ONE test evaluation. Two models: the best pretrained and the best of ours.
# Every extra run here is another look at the frozen split.
FINAL = [
    "v32-resnet34_finetune",     # best overall
    "v28-convnext_big_128",      # best from scratch
]

final_flags = [flag for name in FINAL for flag in ("--only", name)]
posthoc("--split", "both", "--final-test-evaluation",
        "--results", str(OUTPUT_ROOT / "final_test.csv"), *final_flags);

## 6. Read everything

In [ ]:
import pandas as pd

pd.set_option("display.width", 240)
for path in sorted(OUTPUT_ROOT.glob("*.csv")):
    frame = pd.read_csv(path)
    print(f"\n=== {path.name}  ({len(frame)} rows)")
    columns = [c for c in ("run", "split", "px", "tta", "thresholds", "macro_f1",
                           "ci_lower", "ci_upper", "balanced_accuracy",
                           "f1_Loc", "f1_Scratch", "f1_Edge-Loc", "f1_Near-full")
               if c in frame]
    display(frame.sort_values("macro_f1", ascending=False)[columns])

if HAS_DRIVE:
    import shutil
    for path in OUTPUT_ROOT.glob("*.csv"):
        shutil.copy2(path, DRIVE / f"posthoc_{path.name}")
    print("\ncopied to Drive")

## 7. Grad-CAM

`--grid` gives one 3x3 sheet per model covering all nine classes. A red title means that
model misclassified that wafer — keep those rather than cherry-picking hits.

**Transformers are included.** A ViT has no convolution whose axes are spatial, so the
script hooks the last block's LayerNorm, drops the class token, and folds the 49 patch
tokens back into a 7x7 grid.

One caveat worth a line on the slide: on a transformer the weighted sum comes out almost
entirely negative (~2% of positions positive), so the ReLU that makes Grad-CAM readable on a
CNN erases the map. The script falls back to the mean-centred magnitude — the same tensor,
shifted rather than clipped — and only when ReLU leaves nothing. ViT maps are therefore a
slightly different quantity from the CNN ones; do not over-read a direct visual comparison.

In [ ]:
import torch

# All five: the ViT is included now that token models are supported.
# The dilated/control pair is the point -- same architecture, same parameters,
# one sees 63x63 and the other 19x19.
CAM_RUNS = [
    "v32-resnet34_finetune",
    "v32-vit_b_32_finetune",
    "v28-convnext_big_128",
    "v31-dilated_rotation_64",
    "v31-dilated_control_64",
]
CAM_OUTPUT = REPO / "output/gradcam"
CAM_MODELS = [find_checkpoint(name) for name in CAM_RUNS]

# --grid: one 3x3 sheet per model, all nine classes.
command = [sys.executable, str(REPO / "scripts/gradcam.py"),
           "--grid", "--output", str(CAM_OUTPUT)]
for path in CAM_MODELS:
    command += ["--checkpoint", str(path)]
stream(command);

## 8. Show them

In [ ]:
from IPython.display import Image, display

for path in sorted(CAM_OUTPUT.glob("gradcam_grid_*.png")):
    print(path.name)
    display(Image(filename=str(path)))

if HAS_DRIVE:
    target = DRIVE / "gradcam"
    target.mkdir(parents=True, exist_ok=True)
    for path in CAM_OUTPUT.glob("*.png"):
        shutil.copy2(path, target / path.name)
    print("copied to", target)